In [2]:
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point

df = pd.read_csv('ข้อมูลเพาะปลูก.csv')


In [3]:
# ── 1. แปลงวันที่ integer → datetime ──────────────────────────
def parse_thaidate(val):
    try:
        s = str(int(val))
        if len(s) != 8:
            return pd.NaT
        yr = int(s[:4])
        # ถ้า year > 2500 = พ.ศ. → แปลงเป็น ค.ศ.
        if yr > 2500:
            yr -= 543
        return pd.Timestamp(f"{yr}-{s[4:6]}-{s[6:8]}")
    except:
        return pd.NaT

df['plant_date_dt']   = df['plant_date'].apply(parse_thaidate)
df['harvest_date_dt'] = df['produce_da'].apply(parse_thaidate)

# แปลง year_act (พ.ศ. → ค.ศ.)
df['year_ce'] = df['year_act'].apply(lambda y: y - 543 if y > 2500 else y)

In [4]:
# ── 2. คำนวณ growing duration (วัน) ──────────────────────────
df['growing_days'] = (df['harvest_date_dt'] - df['plant_date_dt']).dt.days

# กรอง perennial ที่มีวันที่ผิดปกติ
df['is_annual'] = df['detail_nam'].isin([
    'ข้าวเจ้า','ข้าวเหนียว',
    'ข้าวโพดเลี้ยงสัตว์','มันสำปะหลังโรงงาน'
])

print("Annual crops growing duration (days):")
print(df[df['is_annual']]['growing_days'].describe())

Annual crops growing duration (days):
count    6455.000000
mean      145.510922
std        33.001462
min        90.000000
25%       126.000000
50%       143.000000
75%       158.000000
max       480.000000
Name: growing_days, dtype: float64


In [5]:
# ── 3. Reclassify crop classes ──────────────────────────────
# ปัญหา: tobacco = 2 แปลง, rubber = 30 แปลง
# แนวทาง: ปรับ class scheme ให้สมจริง

class_map = {
    'ข้าวเจ้า':              'rice',
    'ข้าวเหนียว':            'rice',
    'ลำไย':                  'longan',
    'ข้าวโพดเลี้ยงสัตว์':    'corn',
    'มันสำปะหลังโรงงาน':     'cassava',    # ← เพิ่มใหม่! มี 122 แปลง
    'ยางพารา':               'rubber',
    'ยาสูบ':                 'etc',         # ← รวมเข้า etc (2 แปลงเท่านั้น)
    'ปาล์มน้ำมัน':           'etc',
    'มะขาม':                 'etc',
    'หอมแบ่ง(ต้นหอม)':       'etc',
    'ยูคาลิปตัส':            'etc',
    'ฝรั่ง':                 'etc',
    'ฟักทอง':                'etc',
    'กระท่อม':               'etc',
    'มะม่วง':                'etc',
    'ส้มโอ':                 'etc',
    'ผักอื่นๆ':              'etc',
    'กระท้อน':               'etc',
    'ถั่วเขียวผิวมัน':       'etc',
}
df['crop_class'] = df['detail_nam'].map(class_map).fillna('etc')

print("\nRevised class distribution:")
print(df.groupby('crop_class')['act_rai_or'].agg(
    count='count',
    area_rai='sum'
).round(2).sort_values('count', ascending=False))


Revised class distribution:
            count  area_rai
crop_class                 
rice         5720  16329.38
longan        731   2215.06
corn          613   2430.06
cassava       122    623.34
etc            57    182.32
rubber         30    174.96


In [6]:
# ── 4. แปลงเป็น GeoDataFrame ─────────────────────────────────
df_clean = df.dropna(subset=['lat','lng'])
geometry = [Point(lng, lat) for lat, lng
            in zip(df_clean['lat'], df_clean['lng'])]
gdf = gpd.GeoDataFrame(df_clean, geometry=geometry, crs='EPSG:4326')
gdf = gdf.to_crs('EPSG:32647')  # UTM Zone 47N

print(f"\nGeoDataFrame ready: {len(gdf)} parcels")
print(f"CRS: {gdf.crs}")


GeoDataFrame ready: 7269 parcels
CRS: EPSG:32647


In [8]:
# แก้ไข: กรองพิกัดที่ไม่ valid ออกก่อน
# ต.แม่นาเรือ อยู่ใน lat 18.9–19.3, lng 99.5–100.1
valid_mask = (
    df_clean['lat'].between(18.9, 19.3) &
    df_clean['lng'].between(99.5, 100.1)
)
gdf = gdf[valid_mask].copy()
print(f"แถวที่ valid หลังกรองพิกัด: {len(gdf)} (ลบออก {(~valid_mask).sum()} แถว)")

แถวที่ valid หลังกรองพิกัด: 7142 (ลบออก 127 แถว)


In [9]:
# ── 5. Export แยกตาม use case ─────────────────────────────────

# 5a. Training points สำหรับ Phase 1 (RF classification)
#     ใช้ปี 2021–2022 เป็น training | ปี 2023 เป็น validation
gdf_train = gdf[gdf['year_ce'].isin([2021, 2022])]
gdf_val   = gdf[gdf['year_ce'] == 2023]
gdf_train.to_file('training_parcels_2021_2022.shp')
gdf_val.to_file('validation_parcels_2023.shp')
print(f"\nTraining: {len(gdf_train)} | Validation: {len(gdf_val)}")

# 5b. Phenology calendar สำหรับ Phase 2 (DTW template)
#     เฉพาะ annual crops ที่มีวันที่น่าเชื่อถือ
pheno = gdf[gdf['is_annual'] & (gdf['growing_days'] > 30) & (gdf['growing_days'] < 365)]
pheno_cal = pheno.groupby(['crop_class','year_ce']).agg(
    mean_plant_doy  = ('plant_date_dt',   lambda x: x.dt.dayofyear.mean()),
    std_plant_doy   = ('plant_date_dt',   lambda x: x.dt.dayofyear.std()),
    mean_harvest_doy= ('harvest_date_dt', lambda x: x.dt.dayofyear.mean()),
    mean_duration   = ('growing_days',    'mean'),
    n_parcels       = ('act',             'count'),
    total_area_rai  = ('act_rai_or',      'sum'),
).round(1).reset_index()
pheno_cal.to_csv('phenology_calendar_by_crop_year.csv', index=False)
print("\nPhenology calendar (DOY = Day of Year):")
print(pheno_cal.to_string())

# 5c. Crop area per zone สำหรับ Phase 3 (demand estimation)
#     ต้องการ zone boundary ก่อน แต่ prepare ข้อมูลพร้อมไว้
area_summary = gdf.groupby(['year_ce','crop_class'])['act_rai_or'].sum()
area_summary_m2 = (area_summary * 1600).round(0)  # 1 ไร่ = 1,600 m²
area_summary_m2.to_csv('crop_area_m2_by_year.csv')
print("\nCrop area (m²) by year:")
print(area_summary_m2.unstack(fill_value=0))

C:\Users\mpdox\AppData\Local\Temp\ipykernel_26184\192282442.py:7: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf_train.to_file('training_parcels_2021_2022.shp')
C:\Users\mpdox\AppData\Roaming\Python\Python314\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'plant_date_dt' to 'plant_da_1'
  ogr_write(
C:\Users\mpdox\AppData\Roaming\Python\Python314\site-packages\pyogrio\raw.py:733: RuntimeWarning: Field plant_da_1 created as String field, though DateTime requested.
  ogr_write(
C:\Users\mpdox\AppData\Roaming\Python\Python314\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'harvest_date_dt' to 'harvest_da'
  ogr_write(
C:\Users\mpdox\AppData\Roaming\Python\Python314\site-packages\pyogrio\raw.py:733: RuntimeWarning: Field harvest_da created as String field, though DateTime requested.
  ogr_write(
C:\Users\mpdox\AppData\Roaming\Python\Python314\site-packages\pyog


Training: 3556 | Validation: 2177

Phenology calendar (DOY = Day of Year):
   crop_class  year_ce  mean_plant_doy  std_plant_doy  mean_harvest_doy  mean_duration  n_parcels  total_area_rai
0     cassava     2020           121.7           55.6              57.5          301.8         15            61.6
1     cassava     2021           120.5           63.1             134.4          300.7         42           262.4
2     cassava     2022           123.6           28.3             145.4          274.5         13            58.2
3     cassava     2023           107.2           64.5             129.0          313.8          5            20.2
4        corn     2020           186.5           68.9             254.1          120.9        103           431.1
5        corn     2021           171.6           43.9             269.1          118.3        211           913.3
6        corn     2022           159.1           46.8             270.4          119.3        136           518.8
7        cor

In [35]:
import rasterio
import numpy as np
import geopandas as gpd
from rasterio.sample import sample_gen

# ── 1. เช็กว่า geometry เป็น Point หรือ Polygon ──────────────
gdf_in = gpd.read_file('training_parcels_2021_2022.shp')
gdf_in = gdf_in.to_crs('EPSG:32647')
print("Geometry type:", gdf_in.geometry.geom_type.value_counts().to_dict())

# ── 2. เช็ก nodata value ของ raster ─────────────────────────
with rasterio.open('S2_drySeason_composite_2022.tif') as src:
    print(f"\nS2 nodata value: {src.nodata}")
    print(f"S2 dtype       : {src.dtypes[0]}")
    # อ่าน band 1 และดูการกระจายของค่า
    band1 = src.read(1)
    print(f"S2 band1 - NaN count  : {np.isnan(band1).sum()}")
    print(f"S2 band1 - Zero count : {(band1==0).sum()}")
    print(f"S2 band1 - Valid count: {(band1>0).sum()}")
    total = band1.size
    valid_pct = (band1>0).sum() / total * 100
    print(f"S2 band1 - Valid %    : {valid_pct:.1f}%")

with rasterio.open('S1_fullYear_weekly_2022.tif') as src:
    print(f"\nS1 nodata value: {src.nodata}")
    print(f"S1 dtype       : {src.dtypes[0]}")
    band1 = src.read(1)  # VV week 1
    print(f"S1 VV band - NaN count   : {np.isnan(band1).sum()}")
    print(f"S1 VV band - Zero count  : {(band1==0).sum()}")
    valid_pct_s1 = (~np.isnan(band1) & (band1!=0)).sum() / band1.size * 100
    print(f"S1 VV band - Valid %     : {valid_pct_s1:.1f}%")

# ── 3. ทดสอบ extract ค่าที่จุดตรงกลาง raster ────────────────
with rasterio.open('S2_drySeason_composite_2022.tif') as src:
    cx = (src.bounds.left + src.bounds.right)  / 2
    cy = (src.bounds.bottom + src.bounds.top) / 2
    test_val = list(sample_gen(src, [(cx, cy)]))[0]
    print(f"\nS2 value at raster center {cx:.0f},{cy:.0f}:")
    print(f"  {test_val[:5]}  ← 0 หรือ NaN = masked")

Geometry type: {'Point': 3556}

S2 nodata value: None
S2 dtype       : float32
S2 band1 - NaN count  : 180981
S2 band1 - Zero count : 0
S2 band1 - Valid count: 117525
S2 band1 - Valid %    : 39.4%

S1 nodata value: None
S1 dtype       : float32
S1 VV band - NaN count   : 726378
S1 VV band - Zero count  : 0
S1 VV band - Valid %     : 39.1%

S2 value at raster center 589320,2110370:
  [ 523.  797.  987. 1467. 2250.]  ← 0 หรือ NaN = masked


In [2]:
# ═══════════════════════════════════════════════════════════════
# RESTART KERNEL ก่อน แล้วรัน cell นี้ cell เดียว
# ═══════════════════════════════════════════════════════════════
import geopandas as gpd
import rasterio
import numpy as np
import pandas as pd
from rasterio.sample import sample_gen
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import joblib, json

# ── 0. Load parcels + filter to raster extent ────────────────
gdf = gpd.read_file('training_parcels_2021_2022.shp').to_crs('EPSG:32647')
gdf['crop_class'] = gdf['crop_class'].replace('rubber', 'etc')

class_labels = {'rice':0, 'corn':1, 'cassava':2, 'longan':3, 'etc':4}
gdf['class_id'] = gdf['crop_class'].map(class_labels)
gdf = gdf.dropna(subset=['class_id'])
gdf['class_id'] = gdf['class_id'].astype(int)

with rasterio.open('S2_drySeason_composite_2022.tif') as src:
    rb = src.bounds

gdf = gdf[
    gdf.geometry.x.between(rb.left,  rb.right) &
    gdf.geometry.y.between(rb.bottom, rb.top)
].reset_index(drop=True)

coords = list(zip(gdf.geometry.x, gdf.geometry.y))
y      = gdf['class_id'].values
print(f"Parcels: {len(gdf)}")
print(gdf['crop_class'].value_counts())

# ── 1. Extract raster values ──────────────────────────────────
def extract(tif_path, coords, prefix, band_idx=None):
    with rasterio.open(tif_path) as src:
        arr = np.array(list(sample_gen(src, coords)), dtype=np.float32)
    if band_idx is not None:
        arr = arr[:, band_idx]
    return pd.DataFrame(arr, columns=[f"{prefix}_{i}" for i in range(arr.shape[1])])

# S2: ทุก band
s2 = extract('S2_drySeason_composite_2022.tif', coords, 'S2')

# S1: VV + VH ของ week 1–16 เท่านั้น (dry season, 32 bands)
N = 6
vv_idx = [w*N+0 for w in range(16)]
vh_idx = [w*N+1 for w in range(16)]
s1 = extract('S1_fullYear_weekly_2022.tif', coords, 'S1',
             band_idx=vv_idx + vh_idx)

print(f"\nS2 shape: {s2.shape} | S1 shape: {s1.shape}")
print(f"S2 NaN%: {s2.isnull().mean().mean()*100:.1f}%")
print(f"S1 NaN%: {s1.isnull().mean().mean()*100:.1f}%")

# ── แทน Step 2 ทั้งหมด ────────────────────────────────────────

X_raw = pd.concat([s2, s1], axis=1)

print(f"X_raw shape: {X_raw.shape}")
print(f"S2 NaN % per column: {s2.isnull().mean().describe()}")
print(f"S1 NaN % per column: {s1.isnull().mean().describe()}")
print(f"Rows ที่ S2 ทุก band เป็น NaN: {s2.isnull().all(axis=1).sum()}")
print(f"Rows ที่ S2 มีค่าใช้ได้: {s2.notna().all(axis=1).sum()}")

# Impute ด้วย pandas — ไม่ drop columns
col_medians = X_raw.median()        # median ของแต่ละ column (ข้ามทุก row)
X = X_raw.fillna(col_medians)       # แทน NaN ด้วย median

# ถ้า column ใด median ยังเป็น NaN (all-NaN column) → fill ด้วย 0
X = X.fillna(0)

print(f"\nAfter impute shape : {X.shape}")
print(f"Remaining NaN      : {X.isnull().sum().sum()}")
print(f"S2 band1 range     : {X[s2_cols[0]].min():.1f} – {X[s2_cols[0]].max():.1f}")
print(f"S1 VV wk1 range    : {X[s1_cols[0]].min():.2f} – {X[s1_cols[0]].max():.2f}")

# ตรวจ: ถ้า S2 ทุก row เป็น 0 หมด = S2 ใช้ไม่ได้ → ต้องตัดออก
s2_all_zero = (X[s2_cols] == 0).all(axis=1).mean()
print(f"\nRows ที่ S2 เป็น 0 ทั้งหมด: {s2_all_zero*100:.1f}%")
if s2_all_zero > 0.9:
    print("⚠️  S2 ใช้ไม่ได้สำหรับจุดเหล่านี้ — ตัด S2 ออก ใช้ S1 เท่านั้น")
    X = X[s1_cols]
    print(f"→ ใช้ S1 only: {X.shape}")

Parcels: 2830
crop_class
rice       2443
longan      176
corn        174
etc          21
cassava      16
Name: count, dtype: int64

S2 shape: (2830, 14) | S1 shape: (2830, 32)
S2 NaN%: 1.5%
S1 NaN%: 44.7%
X_raw shape: (2830, 46)
S2 NaN % per column: count    1.400000e+01
mean     1.519435e-02
std      5.400623e-18
min      1.519435e-02
25%      1.519435e-02
50%      1.519435e-02
75%      1.519435e-02
max      1.519435e-02
dtype: float64
S1 NaN % per column: count    32.000000
mean      0.446643
std       0.495824
min       0.016254
25%       0.016254
50%       0.016254
75%       1.000000
max       1.000000
dtype: float64
Rows ที่ S2 ทุก band เป็น NaN: 43
Rows ที่ S2 มีค่าใช้ได้: 2787

After impute shape : (2830, 46)
Remaining NaN      : 0
S2 band1 range     : 175.0 – 1722.0
S1 VV wk1 range    : -17.62 – -1.23

Rows ที่ S2 เป็น 0 ทั้งหมด: 0.0%


In [8]:
# ── 4. Train RF ───────────────────────────────────────────────
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, stratify=y, random_state=42)

param_grid = {
    'n_estimators':     [400, 800, 1000, 1100, 1200, 1300],
    'max_depth':        [20, 10, 30,40, None],
    'min_samples_leaf': [2, 5, 7],
    'min_samples_split': [4,5,6],
    'criterion':        ['entropy'],
    'bootstrap':        [ True, False],
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
gs = GridSearchCV(
    RandomForestClassifier(
        class_weight='balanced',
        n_jobs=-1, random_state=42
    ),
    param_grid, cv=cv, scoring='f1_macro', verbose=1
)
gs.fit(X_train, y_train)

Fitting 5 folds for each of 540 candidates, totalling 2700 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestC...ndom_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'bootstrap': [True, False], 'criterion': ['entropy'], 'max_depth': [20, 10, ...], 'min_samples_leaf': [2, 5, ...], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1_macro'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computatio

In [9]:
# ── 5. Evaluate ───────────────────────────────────────────────
y_pred = gs.best_estimator_.predict(X_test)
oa = (y_pred == y_test).mean()
names = list(class_labels.keys())

print(f"\n{'='*50}")
print(f"OA: {oa:.4f}")
print(f"Best params: {gs.best_params_}")
print(f"{'='*50}")
print(classification_report(y_test, y_pred,
      target_names=names, zero_division=0))


OA: 0.9223
Best params: {'bootstrap': True, 'criterion': 'entropy', 'max_depth': 20, 'min_samples_leaf': 2, 'min_samples_split': 4, 'n_estimators': 800}
              precision    recall  f1-score   support

        rice       0.94      0.98      0.96       489
        corn       0.65      0.43      0.52        35
     cassava       1.00      0.67      0.80         3
      longan       0.78      0.71      0.75        35
         etc       1.00      0.50      0.67         4

    accuracy                           0.92       566
   macro avg       0.88      0.66      0.74       566
weighted avg       0.92      0.92      0.92       566



In [11]:
# Feature importance
fi = pd.Series(gs.best_estimator_.feature_importances_,
               index=X.columns).sort_values(ascending=False)
print("\nTop 15 features:")
print(fi.head(15).round(4))
s2_imp = fi[[c for c in fi.index if c.startswith('S2')]].sum()
s1_imp = fi[[c for c in fi.index if c.startswith('S1')]].sum()
print(f"\nS2 importance: {s2_imp:.3f}")
print(f"S1 importance: {s1_imp:.3f}")


Top 15 features:
S2_11    0.0588
S2_12    0.0579
S2_10    0.0465
S2_13    0.0442
S2_9     0.0440
S2_3     0.0411
S2_0     0.0397
S2_1     0.0373
S2_8     0.0361
S2_6     0.0320
S2_2     0.0317
S1_18    0.0309
S2_7     0.0308
S1_2     0.0304
S2_4     0.0301
dtype: float64

S2 importance: 0.560
S1 importance: 0.440


In [14]:
# Save
joblib.dump(gs.best_estimator_, 'rf_model_best.pkl')
joblib.dump(scaler,             'rf_scaler.pkl')
joblib.dump(col_medians,        'col_medians.pkl')
with open('class_labels.json','w') as f:
    json.dump(class_labels, f)
with open('feature_info.json','w') as f:
    json.dump({'s2_cols': s2_cols, 's1_cols': s1_cols,
               'all_cols': all_cols, 'n_feat_s1': 6,
               'vv_idx': vv_idx, 'vh_idx': vh_idx}, f)
print("\n✅ Saved: model, scaler, medians, labels, feature info")


✅ Saved: model, scaler, medians, labels, feature info


In [21]:
"""
=============================================================
Phase 1 — Final: Generate Crop Map with Agricultural Mask
Mae Na Rua Sub-District, Phayao, Northern Thailand
=============================================================
Prerequisites (ต้องมีก่อนรัน):
  - rf_model_best.pkl
  - rf_scaler.pkl
  - col_medians.pkl
  - feature_info.json
  - class_labels.json
  - S2_drySeason_composite_20XX.tif  (2020–2023)
  - S1_fullYear_weekly_20XX.tif      (2020–2023)
  - LDD_landuse.shp
=============================================================
"""

# ── Imports ───────────────────────────────────────────────────
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from rasterio.enums import Resampling
import joblib
import json
import warnings
warnings.filterwarnings('ignore')

# ── Load model และ metadata ───────────────────────────────────
rf      = joblib.load('rf_model_best.pkl')
scaler  = joblib.load('rf_scaler.pkl')
medians = joblib.load('col_medians.pkl')

with open('feature_info.json') as f:
    fi = json.load(f)

with open('class_labels.json') as f:
    class_labels = json.load(f)

all_cols = fi['all_cols']
vv_idx   = fi['vv_idx']
vh_idx   = fi['vh_idx']

# inverse map: id → name
class_names = {v: k for k, v in class_labels.items()}
print("Class labels:", class_labels)
print("Features:", len(all_cols))

# ── LDD Agricultural Mask ────────────────────────────────────
# โหลด 1 ครั้ง ใช้ทุกปี
ldd = gpd.read_file('LDD_landuse.shp')

# LDD L2 codes ที่เป็นเกษตรกรรม
agri_l2 = [20, 21, 22, 23, 24, 25, 26, 27, 29,
           21222, 21225, 22124, 22125, 22222, 22225, 23124]

# ตัด non-crop ออก
exclude_en = ['Teak', 'Fish farm', 'Cattle farm house',
              'Poultry  farm house', 'Swine farm house',
              'Bamboo', 'Rain tree', 'Bur-flower tree']

agri_ldd = ldd[
    ldd['LU_ID_L2'].isin(agri_l2) &
    ~ldd['LU_DES_EN'].isin(exclude_en)
].copy()

print(f"\nAgricultural polygons : {len(agri_ldd)}")
print(f"Agricultural area     : {agri_ldd['RAI'].sum():,} ไร่")


# ── Main Function ─────────────────────────────────────────────
def generate_crop_map(year: int) -> str:
    """
    Generate crop classification map for a given year.

    Parameters
    ----------
    year : int  (2020–2023)

    Returns
    -------
    str : output file path
    """
    s2_path  = f'S2_drySeason_composite_{year}.tif'
    s1_path  = f'S1_fullYear_weekly_{year}.tif'
    out_path = f'crop_map_{year}.tif'

    # ── 1. Load S2 (reference grid) ──────────────────────────
    with rasterio.open(s2_path) as s2_src:
        s2_data   = s2_src.read().astype(np.float32)  # (14, rows, cols)
        profile   = s2_src.profile.copy()
        rows, cols = s2_src.shape
        transform  = s2_src.transform
        crs        = s2_src.crs

    # ── 2. Load S1 และ resample ให้ตรงกับ S2 grid ────────────
    with rasterio.open(s1_path) as s1_src:
        s1_all = s1_src.read(
            out_shape=(s1_src.count, rows, cols),
            resampling=Resampling.bilinear
        ).astype(np.float32)

    # เลือก VV + VH dry season (week 1–16)
    s1_data = s1_all[vv_idx + vh_idx]               # (32, rows, cols)

    print(f"\nYear {year}: S2={s2_data.shape} | S1={s1_data.shape}")

    # ── 3. สร้าง Agricultural Mask ───────────────────────────
    agri_ldd_reproj = agri_ldd.to_crs(crs)

    agri_mask = rasterize(
        [(geom, 1) for geom in agri_ldd_reproj.geometry
         if geom is not None and not geom.is_empty],
        out_shape=(rows, cols),
        transform=transform,
        fill=0,
        dtype=np.uint8
    )

    n_agri = agri_mask.sum()
    pct    = n_agri / (rows * cols) * 100
    print(f"  Agricultural pixels: {n_agri:,} ({pct:.1f}%)")

    # ── 4. Stack + reshape ────────────────────────────────────
    stacked  = np.vstack([s2_data, s1_data])        # (46, rows, cols)
    X_raster = stacked.reshape(len(all_cols), -1).T # (pixels, 46)

    # ── 5. Impute NaN ด้วย medians จาก training ──────────────
    df = pd.DataFrame(X_raster, columns=all_cols)
    df = df.fillna(medians).fillna(0)

    # ── 6. Predict ────────────────────────────────────────────
    X_scaled    = scaler.transform(df.values)
    predictions = rf.predict(X_scaled).astype(np.uint8)

    # ── 7. Apply agricultural mask (อื่นๆ = 255 nodata) ──────
    crop_flat              = np.full(rows * cols, 255, dtype=np.uint8)
    agri_flat              = agri_mask.flatten().astype(bool)
    crop_flat[agri_flat]   = predictions[agri_flat]
    crop_map               = crop_flat.reshape(rows, cols)

    # ── 8. Export GeoTIFF ─────────────────────────────────────
    profile.update(count=1, dtype='uint8', nodata=255)
    with rasterio.open(out_path, 'w', **profile) as dst:
        dst.write(crop_map, 1)

    # ── 9. สรุปพื้นที่ ─────────────────────────────────────────
    pixel_m2    = abs(profile['transform'].a * profile['transform'].e)
    pixel_rai   = pixel_m2 / 1600

    print(f"\n  {'Class':<12} {'Pixels':>10} {'ไร่':>10} {'m²':>14}")
    print(f"  {'-'*50}")

    area_results = {}
    for cid, cname in class_names.items():
        n_px     = int((crop_map == cid).sum())
        rai      = n_px * pixel_rai
        m2       = n_px * pixel_m2
        area_results[cname] = {'pixels': n_px, 'rai': rai, 'm2': m2}
        print(f"  {cname:<12} {n_px:>10,} {rai:>10,.0f} {m2:>14,.0f}")

    total_px = int((crop_map != 255).sum())
    print(f"  {'-'*50}")
    print(f"  {'TOTAL':<12} {total_px:>10,} {total_px*pixel_rai:>10,.0f} {total_px*pixel_m2:>14,.0f}")

    return out_path, area_results


# ── Generate ทุกปี ─────────────────────────────────────────────
all_years_area = {}

for year in [2020, 2021, 2022, 2023]:
    path, areas = generate_crop_map(year)
    all_years_area[year] = areas
    print(f"\n  ✅ {path} saved")

# ── สรุปตาราง crop area ทุกปี ─────────────────────────────────
print("\n" + "="*60)
print("CROP AREA SUMMARY (ไร่) — สำหรับ Phase 3 Demand Estimation")
print("="*60)

rows_summary = []
for year, areas in all_years_area.items():
    for cname, vals in areas.items():
        rows_summary.append({
            'year':    year,
            'crop':    cname,
            'rai':     round(vals['rai'], 1),
            'm2':      round(vals['m2'], 0),
        })

summary_df = pd.DataFrame(rows_summary)
pivot = summary_df.pivot_table(
    index='crop', columns='year', values='rai', aggfunc='sum'
).round(0)

print(pivot.to_string())
summary_df.to_csv('crop_area_by_year.csv', index=False)
print("\n✅ Saved: crop_area_by_year.csv")
print("   → ใช้เป็น input ใน Phase 3: demand estimation")
print("\n✅ Saved: crop_map_2020–2023.tif")
print("   → เปิดตรวจสอบใน QGIS ได้เลย")
print("\n📌 NEXT: Phase 3 — Download ERA5 + CHIRPS → calculate ET0 + NIR/GID")

Class labels: {'rice': 0, 'corn': 1, 'cassava': 2, 'longan': 3, 'etc': 4}
Features: 46

Agricultural polygons : 793
Agricultural area     : 45,994 ไร่

Year 2020: S2=(14, 559, 534) | S1=(32, 559, 534)
  Agricultural pixels: 87,149 (29.2%)

  Class            Pixels        ไร่             m²
  --------------------------------------------------
  rice             71,037     17,759     28,414,800
  corn                  5          1          2,000
  cassava               0          0              0
  longan           16,078      4,020      6,431,200
  etc                  29          7         11,600
  --------------------------------------------------
  TOTAL            87,149     21,787     34,859,600

  ✅ crop_map_2020.tif saved

Year 2021: S2=(14, 559, 534) | S1=(32, 559, 534)
  Agricultural pixels: 87,149 (29.2%)

  Class            Pixels        ไร่             m²
  --------------------------------------------------
  rice             67,165     16,791     26,866,000
  corn         

In [17]:
# สำหรับ Table 1 ใน paper
print("\n=== TABLE 1: Classification Performance ===")
print(f"{'Metric':<12} {'Pinkaeo 2024':>14} {'This study':>12} {'Δ':>8}")
print("-"*50)
print(f"{'OA':<12} {'0.7365':>14} {oa:>12.4f} {(oa-0.7365)*100:>+7.1f}%")
print(f"{'rice F1':<12} {'~0.81':>14} {'0.96':>12}")
print(f"{'corn F1':<12} {'~0.52':>14} {'0.52':>12} {'(same)':>8}")
print(f"{'longan F1':<12} {'~0.72':>14} {'0.73':>12}")
print(f"{'cassava F1':<12} {'N/A':>14} {'0.80':>12} {'(new)':>8}")
print(f"\nNote: corn recall=0.43 — ข้าวโพดช่วงเก็บเกี่ยวมีสเปกตรัมคล้าย bare soil")
print(f"      cassava n_test=3 — ตัวเลขไม่ reliable ทางสถิติ")


=== TABLE 1: Classification Performance ===
Metric         Pinkaeo 2024   This study        Δ
--------------------------------------------------
OA                   0.7365       0.9223   +18.6%
rice F1               ~0.81         0.96
corn F1               ~0.52         0.52   (same)
longan F1             ~0.72         0.73
cassava F1              N/A         0.80    (new)

Note: corn recall=0.43 — ข้าวโพดช่วงเก็บเกี่ยวมีสเปกตรัมคล้าย bare soil
      cassava n_test=3 — ตัวเลขไม่ reliable ทางสถิติ


In [22]:
from sklearn.metrics import classification_report
import pandas as pd

# สร้าง Table 1 สำหรับ paper
report = classification_report(y_test, y_pred,
    target_names=list(class_labels.keys()),
    output_dict=True, zero_division=0)

df_report = pd.DataFrame(report).T.round(3)
df_report.to_csv('table1_classification_report.csv')
print("✅ Saved: table1_classification_report.csv")

# Key numbers สำหรับเขียนใน paper
print("\n=== สำหรับเขียน Section 4.1 ===")
print(f"OA (new 5-class S2+SAR) : {oa:.4f}")
print(f"OA (Pinkaeo 2024, 6-class): 0.7365")
print(f"Improvement              : +{(oa-0.7365)*100:.1f} percentage points")
print(f"Best model               : RF, entropy, depth=20, n=800")
print(f"Training samples         : {len(X_train)}")
print(f"Test samples             : {len(X_test)}")
print(f"\nClasses with limited test support:")
print(f"  cassava : n_test=3  → ระบุใน paper ว่า indicative only")
print(f"  etc     : n_test=4  → รวมกับ rubber → interpret ระวัง")
print(f"\nCorn recall=0.43 explanation:")
print(f"  → S2 dry season ตรงกับ harvest phase ใบแห้ง")
print(f"  → สเปกตรัมคล้าย bare soil → classify ผิดเป็น rice/etc")
print(f"  → แก้ได้ด้วย SAR wet season (Phase 2) ที่จะทำถัดไป")

✅ Saved: table1_classification_report.csv

=== สำหรับเขียน Section 4.1 ===
OA (new 5-class S2+SAR) : 0.9223
OA (Pinkaeo 2024, 6-class): 0.7365
Improvement              : +18.6 percentage points
Best model               : RF, entropy, depth=20, n=800
Training samples         : 2264
Test samples             : 566

Classes with limited test support:
  cassava : n_test=3  → ระบุใน paper ว่า indicative only
  etc     : n_test=4  → รวมกับ rubber → interpret ระวัง

Corn recall=0.43 explanation:
  → S2 dry season ตรงกับ harvest phase ใบแห้ง
  → สเปกตรัมคล้าย bare soil → classify ผิดเป็น rice/etc
  → แก้ได้ด้วย SAR wet season (Phase 2) ที่จะทำถัดไป
